# End-to-End Trading Pipeline: Data, Execution, and Fees

This notebook demonstrates the complete lifecycle of a programmatic trade on Solana.

### The "Cuts" (Who gets paid?)
When you execute a trade in this pipeline, here is the exact fee structure:

1. **Birdeye's Cut:** **$0.00**
   * Birdeye is just a data provider. They do not touch your trade. They make money by charging developers monthly subscriptions for heavy API usage.
2. **Jupiter's Cut:** **$0.00**
   * Jupiter itself does not charge a fee for routing your trade.
   * *However*, the Liquidity Pools that Jupiter routes you through (like Raydium or Orca) charge an **AMM Swap Fee** (usually 0.01% to 0.3%). This fee goes to the Liquidity Providers (people who supplied the NVDAx and USDC to the pool) and is already baked into the Quote you receive.
3. **Solana's Cut:** **~0.000005 SOL (Fraction of a cent)**
   * This is the "Gas Fee". It is paid to the Solana network validators for processing the cryptographic signature and permanently writing your transaction to the ledger.

---


## Step 1 — Setup & Configuration
Load our dependencies and define the asset we want to trade.


In [1]:
import os
import httpx
import base64
import asyncio
from datetime import datetime
from dotenv import load_dotenv

from solders.keypair import Keypair
from solders.transaction import VersionedTransaction
from solana.rpc.async_api import AsyncClient
from solana.rpc.commitment import Processed

# Load environment variables (API keys)
load_dotenv("../.env")
BIRDEYE_API_KEY = os.getenv("BIRDEYE_API_KEY", "")

# Configuration
TARGET_MINT = 'Xsc9qvGR1efVDFGLrVsmkzv3qi45LTBjeUKSPmx9qEh' # NVDAx
USDC_MINT = 'EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v'
RPC_URL = "https://api.mainnet-beta.solana.com"

# We generate a random paper-trading wallet for this run
wallet = Keypair()
wallet_pubkey = str(wallet.pubkey())

print(f"Target Asset: {TARGET_MINT}")
print(f"Paper Wallet: {wallet_pubkey}")
if BIRDEYE_API_KEY:
    print("Birdeye Key:  Loaded!")
else:
    print("Birdeye Key:  MISSING (Check your .env)")


Target Asset: Xsc9qvGR1efVDFGLrVsmkzv3qi45LTBjeUKSPmx9qEh
Paper Wallet: 3d57oZraq3tgoMWGmc9nSnG3R5oBrouxx4TNC3UfmhFj
Birdeye Key:  Loaded!


## Step 2 — The Indexer (Birdeye)
Before we buy, we check the recent trades using Birdeye to establish our baseline market price.


In [2]:
async def fetch_birdeye_data():
    if not BIRDEYE_API_KEY:
        print("Skipping: No Birdeye API Key.")
        return
        
    headers = {"X-API-KEY": BIRDEYE_API_KEY, "x-chain": "solana"}
    
    async with httpx.AsyncClient() as client:
        # Fetch the last 3 trades for context
        res = await client.get(
            f"https://public-api.birdeye.so/defi/txs/token?address={TARGET_MINT}&offset=0&limit=3",
            headers=headers
        )
        
        if res.status_code == 200:
            items = res.json().get("data", {}).get("items", [])
            print("--- RECENT MARKET TRADES (BIRDEYE) ---")
            for tx in items:
                dt = datetime.utcfromtimestamp(tx.get('blockUnixTime', 0)).strftime('%H:%M:%S UTC')
                side = tx.get('side', 'N/A').upper()
                price = tx.get('tokenPrice', 0)
                size = tx.get('base', {}).get('uiAmount', 0)
                print(f"[{dt}] {side:<4} | {size:.4f} NVDAx at ${price:.2f}")
        else:
            print("Birdeye Error:", res.text)

# Run it
await fetch_birdeye_data()


--- RECENT MARKET TRADES (BIRDEYE) ---
[13:42:37 UTC] BUY  | 0.0147 NVDAx at $224.84
[13:42:26 UTC] SELL | 0.0013 NVDAx at $224.79
[13:39:21 UTC] SELL | 0.0019 NVDAx at $224.79


C:\Users\docker\AppData\Local\Temp\ipykernel_19776\1352690650.py:19: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  dt = datetime.utcfromtimestamp(tx.get('blockUnixTime', 0)).strftime('%H:%M:%S UTC')


## Step 3 & 4 — The Executor (Jupiter) & The Ledger (Solana)

### The "ExactOut" Problem & Decimals
On blockchains, there are no fractions. Every token has a `Decimals` value (USDC=6, NVDAx=8). 
To buy 1 NVDAx, you must ask for `100000000` raw units. 

If we ask Jupiter for exactly 1 NVDAx (`swapMode=ExactOut`), it often fails on low-liquidity assets because the math routing engine can't guarantee a clean reverse-route.
**The Solution:** We do a "Probe & Calculate" strategy:
1. **Probe:** Ask Jupiter how much NVDAx we get for exactly $1 USDC (`ExactIn`).
2. **Calculate:** Do the math to find exactly how much USDC is needed to get 1 NVDAx.
3. **Execute:** Send a final quote spending that calculated USDC amount!


In [3]:
async def execute_paper_trade():
    print("--- LIVE JUPITER QUOTE & SIMULATION ---")
    async with httpx.AsyncClient() as client:
        
        # 1. PROBE FOR PRICE
        print("1. PROBE: Asking Jupiter how much NVDAx we get for $1.00 USDC...")
        probe_res = await client.get('https://api.jup.ag/swap/v1/quote', params={
            'inputMint': USDC_MINT, 'outputMint': TARGET_MINT,
            'amount': '1000000', # 1 USDC (6 decimals)
            'swapMode': 'ExactIn', 'slippageBps': 50
        })
        
        if probe_res.status_code != 200:
            print("Probe failed.", probe_res.text)
            return
            
        probe_data = probe_res.json()
        
        # Normalization: Divide raw output by 10^8 (NVDAx decimals)
        nvdax_per_dollar = int(probe_data.get('outAmount', 0)) / 1e8
        print(f"   -> $1.00 USDC buys exactly {nvdax_per_dollar:.6f} NVDAx.")
        
        # 2. CALCULATE
        # If $1 buys 0.0044 shares, then 1 share costs (1 / 0.0044) dollars.
        cost_per_full_nvdax = 1.0 / nvdax_per_dollar
        raw_usdc_needed = int(cost_per_full_nvdax * 1e6) # Convert back to raw 6 decimals
        
        print(f"\n2. CALCULATE: To buy exactly 1.0 NVDAx, we need to spend ${cost_per_full_nvdax:.4f} USDC.")
        print(f"   -> Raw USDC integer to send: {raw_usdc_needed}")
        
        # 3. FINAL QUOTE
        print("\n3. FINAL QUOTE: Executing trade for that exact USDC amount...")
        quote_res = await client.get('https://api.jup.ag/swap/v1/quote', params={
            'inputMint': USDC_MINT, 'outputMint': TARGET_MINT,
            'amount': str(raw_usdc_needed), 
            'swapMode': 'ExactIn', 'slippageBps': 50
        })
        quote_data = quote_res.json()
        final_received = int(quote_data.get('outAmount', 0)) / 1e8
        print(f"   -> Quote Success! We will receive ~{final_received:.6f} NVDAx.")
        
        # 4. BUILD
        print("\n4. Asking Jupiter to construct the raw Solana transaction...")
        swap_res = await client.post('https://api.jup.ag/swap/v1/swap', json={
            'quoteResponse': quote_data,
            'userPublicKey': wallet_pubkey,
            'wrapAndUnwrapSol': True
        })
        
        if swap_res.status_code != 200:
            print("Swap generation failed:", swap_res.text)
            return
            
        swap_data = swap_res.json()
        raw_tx = base64.b64decode(swap_data['swapTransaction'])
        tx = VersionedTransaction.from_bytes(raw_tx)
        print("   -> Transaction successfully built and deserialized.")

        # 5. SIMULATE ON SOLANA LEDGER
        print("\n5. Sending transaction to Solana Mainnet for Simulation...")
        async with AsyncClient(RPC_URL) as rpc:
            sim_res = await rpc.simulate_transaction(tx, commitment=Processed)
            val = sim_res.value
            
            if val.err:
                print("   -> ❌ Simulation Failed! (Expected because our paper wallet has $0 USDC balance)")
                print(f"      Error Code: {val.err}")
                compute_units = val.units_consumed if val.units_consumed else 0
            else:
                print("   -> ✅ Simulation Succeeded!")
                compute_units = val.units_consumed
                
            print(f"\n--- SOLANA NETWORK FEES (FOR EMPIRICAL PAPER) ---")
            print("To properly separate Gas Fees from Asset Price (P_JUP):")
            
            # Solana Base Fee is always 5000 Lamports (0.000005 SOL)
            # Plus priority fees based on Compute Units consumed.
            base_fee_sol = 0.000005 
            print(f"1. Base Signature Fee:    {base_fee_sol:.6f} SOL")
            print(f"2. Compute Units Used:    {compute_units} CU")
            print(f"3. Total Execution Gas:   ~{base_fee_sol:.6f} SOL (Separated from asset cost)")
            
            print(f"\nJupiter Routing Fee:      $0.00")
            print(f"Birdeye Indexing Fee:     $0.00")

# Run it
await execute_paper_trade()


--- LIVE JUPITER QUOTE & SIMULATION ---
1. PROBE: Asking Jupiter how much NVDAx we get for $1.00 USDC...
   -> $1.00 USDC buys exactly 0.004442 NVDAx.

2. CALCULATE: To buy exactly 1.0 NVDAx, we need to spend $225.1000 USDC.
   -> Raw USDC integer to send: 225100000

3. FINAL QUOTE: Executing trade for that exact USDC amount...
   -> Quote Success! We will receive ~0.999851 NVDAx.

4. Asking Jupiter to construct the raw Solana transaction...
   -> Transaction successfully built and deserialized.

5. Sending transaction to Solana Mainnet for Simulation...
   -> ❌ Simulation Failed! (Expected because our paper wallet has $0 USDC balance)
      Error Code: TransactionErrorFieldless.AccountNotFound

--- SOLANA NETWORK FEES (FOR EMPIRICAL PAPER) ---
To properly separate Gas Fees from Asset Price (P_JUP):
1. Base Signature Fee:    0.000005 SOL
2. Compute Units Used:    0 CU
3. Total Execution Gas:   ~0.000005 SOL (Separated from asset cost)

Jupiter Routing Fee:      $0.00
Birdeye Indexing F